<a href="https://colab.research.google.com/github/ProNomanRizvi/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProNomanRizvi/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ProNomanRizvi/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
<br>
This is a **scoring / ranking** task. The goal isn't to classify pages into fixed buckets — it's
to rank all pages by "how urgently does this need a content refresh?" so a limited review team
knows what to look at first. Ranking fits because the real decision is comparative (what's the
top 50 to fix first), not binary.

In [11]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Task type: ranking/scoring — output is a priority score per page, not a fixed label.")

Task type: ranking/scoring — output is a priority score per page, not a fixed label.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
<br>
**Proxy target:** `is_declining_label = (trend_direction == "down")`. This is a proxy, not a
direct observed outcome — "declining" is a category derived from a trend field in the data, not
a client's actual business decision. It stands in for "this page needs attention soon."

In [12]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [13]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship/flyrank-ml-internship/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /c

## 3. Success metric

*One metric you can defend. What number means 'good'?*
<br>
**Success metric: Precision@50.** Of the top 50 pages the model ranks highest, how many are
actually declining? This matches how the output is used — a reviewer only has time to check a
short list, so precision at the top of the ranking matters more than overall accuracy.

In [14]:
import json
res = json.load(open("outputs/model_results.json"))
print("Baseline Precision@50:", res["baseline"]["baseline_precision_at_50"])
print("Model Precision@50:", res["models"]["random_forest"]["precision_at_50"])

Baseline Precision@50: 0.24
Model Precision@50: 0.74


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
<br>
**Unit of analysis: one row = one page.** Each row is a single anonymized page tracked over a
90-day window, with its own search, content, and trend metrics.

In [15]:
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

30000 rows, 45 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
<br>
A fixed if-statement rule (the baseline) only reached Precision@50 = 0.24. The learned model hit
roughly 0.68–0.74 — about 3x better. That gap means the real signal isn't one threshold on one
column; it's an interaction of several weak signals (search volume, CTR, position tier, content
age) that a single rule can't capture but a model can weigh together.

In [16]:
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Lift over baseline: {rf/base:.1f}x")

Lift over baseline: 3.1x


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.